In [47]:
from pyspark.sql import SparkSession

In [48]:
spark = SparkSession.builder\
        .appName('joins')\
        .master('local[*]')\
        .getOrCreate()

In [9]:
orders_data = [
    (1, 201, "laptop", 1200),
    (2, 202, "phone", 800),
    (3, 203, "tablet", 600),
    (4, 201, "mouse", 50),
    (5, 204, "keyboard", 100),
    (6, 205, "monitor", 300),
    (7, 206, "chair", 150)
]

orders_df = spark.createDataFrame(orders_data, ["order_id", "customer_id", "product", "amount"])


customers_data = [
    (201, "Alice", "US"),
    (202, "Bob", "IN"),
    (203, "Charlie", "US"),
    (207, "David", "UK")
]

customers_df = spark.createDataFrame(customers_data, ["customer_id", "name", "country"])


In [6]:
payments_data = [
    (1, "SUCCESS"),
    (2, "FAILED"),
    (3, "SUCCESS"),
    (4, "SUCCESS"),
    (8, "FAILED")
]

payments_df = spark.createDataFrame(payments_data, ["order_id", "payment_status"])
payments_df.show()

+--------+--------------+
|order_id|payment_status|
+--------+--------------+
|       1|       SUCCESS|
|       2|        FAILED|
|       3|       SUCCESS|
|       4|       SUCCESS|
|       8|        FAILED|
+--------+--------------+



In [14]:
joined_df = orders_df.join(customers_df, on="customer_id",how="left")\
                     .select(
                         'order_id',
                         'name',
                         'amount'
                     )

joined_df.show()

+--------+-------+------+
|order_id|   name|amount|
+--------+-------+------+
|       1|  Alice|  1200|
|       2|    Bob|   800|
|       3|Charlie|   600|
|       4|  Alice|    50|
|       5|   NULL|   100|
|       6|   NULL|   300|
|       7|   NULL|   150|
+--------+-------+------+



In [28]:
from pyspark.sql.functions import col
o = orders_df.alias("o")
c = customers_df.alias("c")
joined_df = o.join(c, o.customer_id == c.customer_id,how="left")\
                    .select(
                        col('o.customer_id')
                    ).show()

+-----------+
|customer_id|
+-----------+
|        201|
|        202|
|        203|
|        201|
|        204|
|        205|
|        206|
+-----------+



In [29]:
o = orders_df.alias('o')
c = customers_df.alias('c')

In [35]:
#shows unmatched rows from left table only
joined_df = o.join(c,
                   o.customer_id == c.customer_id,
                   how = 'left_anti'
                  ).show()


+--------+-----------+--------+------+
|order_id|customer_id| product|amount|
+--------+-----------+--------+------+
|       5|        204|keyboard|   100|
|       6|        205| monitor|   300|
|       7|        206|   chair|   150|
+--------+-----------+--------+------+



In [36]:
#shows matched rows from left table only
joined_df = o.join(c,
                   o.customer_id == c.customer_id,
                   how = 'left_semi'
                  ).show()


+--------+-----------+-------+------+
|order_id|customer_id|product|amount|
+--------+-----------+-------+------+
|       1|        201| laptop|  1200|
|       4|        201|  mouse|    50|
|       2|        202|  phone|   800|
|       3|        203| tablet|   600|
+--------+-----------+-------+------+



### MULTI TABLE JOINS

In [37]:
o = orders_df.alias("o")
c = customers_df.alias("c")
p = payments_df.alias("p")

final_df = o.join(
    c, col("o.customer_id") == col("c.customer_id"), "left"
).join(
    p, col("o.order_id") == col("p.order_id"), "left"
).select(
    col("o.order_id"),
    col("c.name"),
    col("o.product"),
    col("o.amount"),
    col("p.payment_status")
)

final_df.show()

+--------+-------+--------+------+--------------+
|order_id|   name| product|amount|payment_status|
+--------+-------+--------+------+--------------+
|       7|   NULL|   chair|   150|          NULL|
|       6|   NULL| monitor|   300|          NULL|
|       5|   NULL|keyboard|   100|          NULL|
|       1|  Alice|  laptop|  1200|       SUCCESS|
|       3|Charlie|  tablet|   600|       SUCCESS|
|       2|    Bob|   phone|   800|        FAILED|
|       4|  Alice|   mouse|    50|       SUCCESS|
+--------+-------+--------+------+--------------+



#### Join all 3 tables and return:

- order_id
- customer_id
- name
- payment_status

In [42]:
o = orders_df.alias('o')
c = customers_df.alias('c')
p = payments_df.alias('p')

joined_df = \
o.join(
    c,
    o.customer_id == c.customer_id,
    how = "inner"
)\
.join(
    p,
    o.order_id == p.order_id,
    how = "inner"
)\
.select(
    col('o.order_id'),
    col('c.customer_id'),
    col('c.name'),
    col('p.payment_status')
).show()

+--------+-----------+-------+--------------+
|order_id|customer_id|   name|payment_status|
+--------+-----------+-------+--------------+
|       1|        201|  Alice|       SUCCESS|
|       2|        202|    Bob|        FAILED|
|       3|        203|Charlie|       SUCCESS|
|       4|        201|  Alice|       SUCCESS|
+--------+-----------+-------+--------------+



#### 👉 Orders that have NO payment record

In [43]:
joined_df = \
    o.join(
        p,
        o.order_id == p.order_id,
        how = "left_anti"
    ).show()

+--------+-----------+--------+------+
|order_id|customer_id| product|amount|
+--------+-----------+--------+------+
|       5|        204|keyboard|   100|
|       6|        205| monitor|   300|
|       7|        206|   chair|   150|
+--------+-----------+--------+------+



In [49]:
orders_big = orders_df.crossJoin(spark.range(0, 1000)).drop("id")
customers_small = customers_df

In [51]:
normal_join = orders_big.join(
    customers_small,
    "customer_id",
    "inner"
)

normal_join.show()

+-----------+--------+-------+------+-----+-------+
|customer_id|order_id|product|amount| name|country|
+-----------+--------+-------+------+-----+-------+
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201

In [53]:
from pyspark.sql.functions import broadcast
normal_join = orders_big.join(
    broadcast(customers_small),
    "customer_id",
    "inner"
)

normal_join.show()

+-----------+--------+-------+------+-----+-------+
|customer_id|order_id|product|amount| name|country|
+-----------+--------+-------+------+-----+-------+
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201|       1| laptop|  1200|Alice|     US|
|        201